# goals_foragst — draw — XGBoost, no segmentation

The goals for/against space + market draw anchor, XGBoost, no segmentation. Reproduces the goals_foragst_draw_kmeans_match_lr edge without the KMeans region filter.

In [1]:
# --- SETUP -------------------------------------------------------------------
GROUP = "major"   # league group: which <group>/ ABT to validate
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from xgboost import XGBClassifier

from evaluation.model_check import run_check, evaluate

abt = pd.read_parquet(f"C:/Users/twluc/frame/01_data/03_abt/{GROUP}/abt.parquet")

In [2]:
# --- SPACE (modifiable) — the model space, defined in one place --------------
# The features the bet-signal model reads. Change this cell (only) to move the model to another space.
FEATURES = [
    "hmt_season_goals_for_avg",
    "hmt_season_goals_agst_avg",
    "awt_season_goals_for_avg",
    "awt_season_goals_agst_avg",
    "mrkt_draw_impl",
]

In [3]:
# --- TARGET ------------------------------------------------------------------
# OUTCOME is the 0/1 event we bet on. IMPLIED is the market-consensus implied probability (mrkt),
# used for both the profitability check and the ROI — one odds source, never mixed.
OUTCOME = "t_draw_flg"          # 1 when the match ended in a draw
IMPLIED = "mrkt_draw_impl"      # implied draw prob = 1 / mrkt draw odds (profitability + betting)

In [4]:
# --- ABT FILTERING -----------------------------------------------------------
# Keep matches where both teams have enough history for stable season averages; drop rows missing any
# feature or the target.
MIN_GAME_NUMBER = 8
matches = abt[
    (abt["hmt_game_number"] > MIN_GAME_NUMBER)
    & (abt["awt_game_number"] > MIN_GAME_NUMBER)
].dropna(subset=FEATURES + [IMPLIED, OUTCOME, "league", "season"]).copy()

print(f"matches: {len(matches)}   seasons: {sorted(matches['season'].unique())}   "
      f"leagues: {sorted(matches['league'].unique())}")

matches: 5533   seasons: ['2223', '2324', '2425', '2526']   leagues: ['england', 'france', 'germany', 'italy', 'spain']


In [5]:
# --- SEGMENTER — NONE (this notebook tests the model WITHOUT segmentation) ---
# The segmenter contract still holds: segmenter(train) -> labeler ; labeler(rows) -> Series where
# NaN means do-not-bet. Here every row is labelled 0 (never NaN), so we bet everywhere and the MODEL
# alone decides where (model_prob > implied + buffer) — no KMeans region filter. This isolates the
# model + space effect from the segmentation used in the *_kmeans_* notebooks.
def segmenter(train):
    return lambda rows: pd.Series(0, index=rows.index)

In [6]:
# --- MODEL (modifiable) — XGBoost -------------------------------------------
# Shallow, regularized gradient-boosted trees. max_depth sets the interaction order: the draw edge is
# an interaction between the team goal/points profiles and the market draw line, so depth >= 2 is
# needed (depth-1 stumps, purely additive, fail). Strong regularization + row/column subsampling keep
# the thin 4-season signal from overfitting the training seasons.
def make_model():
    return XGBClassifier(max_depth=3, n_estimators=120, learning_rate=0.05,
                         subsample=0.7, colsample_bytree=0.8, min_child_weight=10,
                         reg_lambda=3.0, reg_alpha=0.5, eval_metric="logloss",
                         n_jobs=4, verbosity=0)

BUFFER = 0.05   # only bet when the model probability beats the implied one by at least this margin

In [7]:
_ = run_check(matches, features=FEATURES, outcome=OUTCOME, implied=IMPLIED,
              segmenter=segmenter, make_model=make_model, buffer=BUFFER,
              source="goals_foragst_draw_xgboost_match", space="goals_foragst", target="draw",
              abt_filter="major, gn>8, dropna",
              segmentation="none (bet all)", segment_filter="none")

goals_foragst_draw_xgboost_match   pooled +0.164 (673 bets)   [mrkt]

GATE 1  LOSO — ROI by test season           PASS  (+4/4 seasons)
    2223 +0.016    2324 +0.201    2425 +0.136    2526 +0.321

GATE 2  league — ROI by league / drop best   PASS
    spain +0.276    germany +0.272    england +0.209    france +0.093    italy +0.016
    pooled without best league (germany): +0.135

GATE 3  walk-forward — ROI by test season    PASS
    2425 -0.027    2526 +0.321


In [8]:
# EXTRA CHECK — knob robustness: sweep tree depth x betting buffer. A real edge survives a RANGE of
# both, forming a contiguous plateau; one that appears at a single (depth, buffer) is fragile. Uses
# evaluate() so nothing is logged to the registry.
print(" depth  buf   pooled   profit   bets  gates")
for depth in (2, 3, 4):
    for buf in (0.02, 0.03, 0.04, 0.05, 0.06):
        mk = lambda d=depth: XGBClassifier(max_depth=d, n_estimators=120, learning_rate=0.05,
                 subsample=0.7, colsample_bytree=0.8, min_child_weight=10,
                 reg_lambda=3.0, reg_alpha=0.5, eval_metric="logloss", n_jobs=4, verbosity=0)
        r = evaluate(matches, features=FEATURES, outcome=OUTCOME, implied=IMPLIED,
                     segmenter=segmenter, make_model=mk, buffer=buf)
        if r["empty"]:
            print(f" {depth:<5}  {buf:.2f}  no bets"); continue
        b = r["bets"]; ng = int(r["g1"]) + int(r["g2"]) + int(r["g3"])
        print(f" {depth:<5}  {buf:.2f} {r['pooled']:+7.3f} {(b['won']-b['implied']).sum():+8.2f} {len(b):>5}   {ng}/3")
    print()

 depth  buf   pooled   profit   bets  gates


 2      0.02  +0.010    +3.06  1187   1/3


 2      0.03  +0.084   +18.74   871   3/3


 2      0.04  +0.083   +13.87   650   2/3


 2      0.05  +0.129   +14.77   448   3/3


...

 4      0.02  +0.067   +26.46  1526   2/3


 4      0.03  +0.067   +21.57  1253   3/3


 4      0.04  +0.100   +26.71  1033   3/3


 4      0.05  +0.134   +29.27   840   3/3


 4      0.06  +0.140   +23.89   659   3/3

